In [1]:
from qvarnet.models.exponential import LogExponentialMLPwithPenalty, LogExponentialMLPwithGaussianPenalty, LogAnalyticWavefunction
from qvarnet.train import train
import jax
import matplotlib.pyplot as plt
import numpy as np
import jax.numpy as jnp
from qvarnet.probability import build_prob_fn
from qvarnet.sampling_step import sample_and_process
from qvarnet.training_step import energy_and_grads
from qvarnet.hamiltonian.continuous import HarmonicOscillatorHamiltonian

NOT IMPLEMENTED YET: PairLogExponentialMLPwithGaussianPenalty


# SETUP

In [2]:
N_PARTICLES = 2
DIM = 1
N_CHAINS = 5_000
DoF = N_PARTICLES * DIM
SHAPE = (N_CHAINS, DoF)
EPOCHS = 100
N_STEPS = 50

model = LogExponentialMLPwithPenalty(
        architecture=[N_PARTICLES, 6, 1],
        hidden_activation=jax.nn.tanh,
        kernel_init=jax.nn.initializers.normal(stddev=.01),
        bias_init=jax.nn.initializers.normal(stddev=.01),
    )

In [3]:
def _cm_relative(x, n_particles, n_dim):
    """Transform to CM coordinates.

    r_i = x_i - X_CM  for i < N
    r_N = X_CM
    """
    shape = x.shape[:-1]
    r = x.reshape(*shape, n_particles, n_dim)  # (..., n_particles, n_dim)
    cm = r.mean(axis=-2, keepdims=True)        # (..., 1, n_dim)
    r = r - cm                                 # (..., n_particles, n_dim)
    r = r.at[..., -1, :].set(cm.squeeze(-2))  # last particle ← X_CM
    return r.reshape(*shape, n_particles * n_dim)

# def _cm_relative(x,n_particles, n_dim):
#     return x

In [4]:
def compute_with_transform_in_model():
    def model_apply(params, x_batch):
        return model.apply(params, _cm_relative(x_batch, n_particles=N_PARTICLES, n_dim=DIM))
    
    params = model.init(jax.random.PRNGKey(0), jnp.zeros(SHAPE))
    positions = jnp.zeros(SHAPE)  # warm-restart buffer

    for epoch in range(EPOCHS):
        x_batch, positions, _ = sample_and_process(
                                    key=jax.random.PRNGKey(epoch),
                                    prob_fn=build_prob_fn(model_apply, True),
                                    prob_params=params,
                                    init_positions=positions,  # carry over last positions
                                    step_size=0.5,
                                    n_chains=N_CHAINS,
                                    DoF=DoF,
                                    n_steps=N_STEPS,
                                    burn_in=N_STEPS//2,
                                    thinning=1,
                                    PBC=1.0,
                                    is_log_prob=True)
        
        energy, s_energy, grads = energy_and_grads(hamiltonian=HarmonicOscillatorHamiltonian(),
                                                    params=params,
                                                    batch=x_batch,
                                                    model_apply=model_apply,
                                                    is_log_model=True)
        
        params = jax.tree.map(lambda p, g: p - 0.01 * g, params, grads)

        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Energy = {energy:.6f} ± {s_energy:.6f}")

def compute_outside_model():
    def model_apply(params, x_batch):
        return model.apply(params, x_batch)

    params = model.init(jax.random.PRNGKey(0), jnp.zeros(SHAPE))
    positions = jnp.zeros(SHAPE)  # warm-restart buffer

    for epoch in range(EPOCHS):
        x_batch, positions, _ = sample_and_process(
                                    key=jax.random.PRNGKey(epoch),
                                    prob_fn=build_prob_fn(model_apply, True),
                                    prob_params=params,
                                    init_positions=positions,  # carry over last positions
                                    step_size=0.5,
                                    n_chains=N_CHAINS,
                                    DoF=DoF,
                                    n_steps=N_STEPS,
                                    burn_in=N_STEPS//2,
                                    thinning=1,
                                    PBC=1.0,
                                    is_log_prob=True)

        x_batch = _cm_relative(x_batch, n_particles=N_PARTICLES, n_dim=DIM)
        
        energy, s_energy, grads = energy_and_grads(hamiltonian=HarmonicOscillatorHamiltonian(),
                                                    params=params,
                                                    batch=x_batch,
                                                    model_apply=model_apply,
                                                    is_log_model=True)
        
        params = jax.tree.map(lambda p, g: p - 0.01 * g, params, grads)

        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Energy = {energy:.6f} ± {s_energy:.6f}")

In [5]:
compute_with_transform_in_model()

Epoch 0: Energy = 1.194743 ± 1.055363
Epoch 10: Energy = 1.193714 ± 1.051879
Epoch 20: Energy = 1.188072 ± 0.988284
Epoch 30: Energy = 1.184496 ± 1.021678
Epoch 40: Energy = 1.197007 ± 0.988913
Epoch 50: Energy = 1.182523 ± 1.033448
Epoch 60: Energy = 1.179667 ± 1.104315
Epoch 70: Energy = 1.188994 ± 1.006240
Epoch 80: Energy = 1.186341 ± 1.030668
Epoch 90: Energy = 1.186726 ± 1.004931


In [6]:
compute_outside_model()

Epoch 0: Energy = 1.287338 ± 0.822456
Epoch 10: Energy = 1.293218 ± 0.823164
Epoch 20: Energy = 1.287779 ± 0.828181
Epoch 30: Energy = 1.298071 ± 0.840977
Epoch 40: Energy = 1.309268 ± 0.834196
Epoch 50: Energy = 1.307753 ± 0.846499
Epoch 60: Energy = 1.313627 ± 0.848916
Epoch 70: Energy = 1.316679 ± 0.841771
Epoch 80: Energy = 1.325609 ± 0.848827
Epoch 90: Energy = 1.325606 ± 0.865145
